# Lambda = 2.0: Rerun of VQE experiments

Notebook created by HLD for the work arXiv: 2503.13368 [quant-ph, hep-th]

In this notebook, we rerun all those VQE experiments with lower bounds. 

In [16]:
import numpy as np
import pylab
import matplotlib.pyplot as plt
from qiskit.circuit.library import TwoLocal, EfficientSU2
import time
import sys
sys.path.append('../../utility')
from vqe_run import *
from qc_ansatze import *

In [7]:
ansatz_0a = TL_ansatz(6, 'ry', 'crx', "circular", 1)
ansatz_0d = TL_ansatz(6,['ry','y'], 'crx', 'circular', 1)

Circuit ansatz with 18 parameters
Circuit ansatz with 18 parameters


In [12]:
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import SPSA
from qiskit_algorithms.utils import algorithm_globals 
from qiskit_aer.primitives import Estimator as AerEstimator
import warnings
warnings.filterwarnings("ignore")

#storing values
counts = []
values = []
def store_intermediate_result(eval_count, parameters, mean, std):
    counts.append(eval_count)
    values.append(mean)
    
def run_qve_w_specified_optimizer(operator, optimizer, ansatz, seed = 170, iterations = 250):
    algorithm_globals.random_seed = seed
    noiseless_estimator = AerEstimator(
        run_options={"seed": seed, "shots": 1024},
        transpile_options={"seed_transpiler": seed},)
    opt = optimizer(maxiter = iterations)
    vqe = VQE(noiseless_estimator, ansatz, optimizer=opt, callback=store_intermediate_result)
    result = vqe.compute_minimum_eigenvalue(operator).eigenvalue.real
    print(f"VQE result: {result:.5f}")
    return result

# lambda = 0.2

In [29]:
Hpauli =[('IIIIII', 6.15),
 ('IIIIIZ', -0.5),
 ('IIIIZI', -0.5),
 ('IIIZII', -0.5),
 ('IIZIII', -0.5),
 ('IXXIXX', -0.05),
 ('IZIIII', -0.5),
 ('XIXXIX', -0.05),
 ('XXIXXI', -0.05),
 ('ZIIIII', -0.5)]

from qiskit.quantum_info import SparsePauliOp
from qiskit_algorithms import NumPyEigensolver
H4q = SparsePauliOp.from_list(Hpauli)

# exactly diagonalize the system using numpy routines
solver = NumPyEigensolver(k=4)
exact_solution = solver.compute_eigenvalues(H4q)
print("Exact Result of qubit hamiltonian:", np.real(exact_solution.eigenvalues))
E_exact = np.round(np.real(exact_solution.eigenvalues)[0],5)
E_exact

Exact Result of qubit hamiltonian: [3.14807787 4.14674965 4.14674965 4.14674965]


3.14808

In [31]:
r_res=[]
seeds = [18, 28, 38, 48, 58, 68, 78, 88, 188]
for i in range(len(seeds)):
    print(f'At step {i}, with {seeds[i]}')
    counts = []
    values = []
    t0 = time.time()
    result = run_qve_w_specified_optimizer(H4q, COBYLA, ansatz_0a, seeds[i], iterations = 250)
    t1 = time.time()
    print(f'Length of this optimization {len(values)}, time taken = {np.round(t1-t0,3)} \n')
    counts_a = counts
    values_a = values 
    r_res.append(pd.DataFrame({f'seed_{seeds[i]}': values_a}))

At step 0, with 18
VQE result: 3.14707
Length of this optimization 211, time taken = 1.436 

At step 1, with 28
VQE result: 3.14297
Length of this optimization 219, time taken = 1.498 

At step 2, with 38
VQE result: 3.14551
Length of this optimization 211, time taken = 1.57 

At step 3, with 48
VQE result: 3.14863
Length of this optimization 194, time taken = 1.319 

At step 4, with 58
VQE result: 3.14727
Length of this optimization 180, time taken = 1.232 

At step 5, with 68
VQE result: 3.14473
Length of this optimization 204, time taken = 1.347 

At step 6, with 78
VQE result: 3.15000
Length of this optimization 205, time taken = 1.406 

At step 7, with 88
VQE result: 3.15332
Length of this optimization 226, time taken = 1.535 

At step 8, with 188
VQE result: 3.14766
Length of this optimization 196, time taken = 1.375 



In [33]:
df1 = pd.concat([r_res[i] for i in range(len(r_res))], axis = 1)
df1.to_csv('results_seeds/l2_l02_tl_Ry_c_cobyla_seeds.csv')

In [35]:
r_res=[]
seeds = [18, 28, 38, 48, 58, 68, 78, 88, 188]
for i in range(len(seeds)):
    print(f'At step {i}, with {seeds[i]}')
    counts = []
    values = []
    t0 = time.time()
    result = run_qve_w_specified_optimizer(H4q, COBYLA, ansatz_0d, seeds[i], iterations = 250)
    t1 = time.time()
    print(f'Length of this optimization {len(values)}, time taken = {np.round(t1-t0,3)} \n')
    counts_a = counts
    values_a = values 
    r_res.append(pd.DataFrame({f'seed_{seeds[i]}': values_a}))

At step 0, with 18
VQE result: 3.14707
Length of this optimization 211, time taken = 1.572 

At step 1, with 28
VQE result: 3.14297
Length of this optimization 219, time taken = 1.592 

At step 2, with 38
VQE result: 3.14551
Length of this optimization 211, time taken = 1.531 

At step 3, with 48
VQE result: 3.14863
Length of this optimization 194, time taken = 1.432 

At step 4, with 58
VQE result: 3.14727
Length of this optimization 180, time taken = 1.467 

At step 5, with 68
VQE result: 3.14473
Length of this optimization 204, time taken = 1.507 

At step 6, with 78
VQE result: 3.15000
Length of this optimization 205, time taken = 1.482 

At step 7, with 88
VQE result: 3.15332
Length of this optimization 226, time taken = 1.658 

At step 8, with 188
VQE result: 3.14766
Length of this optimization 196, time taken = 1.456 



In [37]:
df2 = pd.concat([r_res[i] for i in range(len(r_res))], axis = 1)
df2.to_csv('results_seeds/l2_l02_tl_RyY_c_cobyla_seeds.csv')

# lambda = 0.5

In [10]:
Hpauli =[('IIIIII', 6.375),
 ('IIIIIZ', -0.5),
 ('IIIIZI', -0.5),
 ('IIIZII', -0.5),
 ('IIZIII', -0.5),
 ('IXXIXX', -0.125),
 ('IZIIII', -0.5),
 ('XIXXIX', -0.125),
 ('XXIXXI', -0.125),
 ('ZIIIII', -0.5)]

from qiskit.quantum_info import SparsePauliOp
from qiskit_algorithms import NumPyEigensolver
H4q = SparsePauliOp.from_list(Hpauli)

# exactly diagonalize the system using numpy routines
solver = NumPyEigensolver(k=4)
exact_solution = solver.compute_eigenvalues(H4q)
print("Exact Result of qubit hamiltonian:", np.real(exact_solution.eigenvalues))
E_exact = np.round(np.real(exact_solution.eigenvalues)[0],5)
E_exact

Exact Result of qubit hamiltonian: [3.36254139 4.35352431 4.35352431 4.35352431]


3.36254

In [20]:
r_res=[]
seeds = [18, 28, 38, 48, 58, 68, 78, 88, 188]
for i in range(len(seeds)):
    print(f'At step {i}, with {seeds[i]}')
    counts = []
    values = []
    t0 = time.time()
    result = run_qve_w_specified_optimizer(H4q, COBYLA, ansatz_0a, seeds[i], iterations = 250)
    t1 = time.time()
    print(f'Length of this optimization {len(values)}, time taken = {np.round(t1-t0,3)} \n')
    counts_a = counts
    values_a = values 
    r_res.append(pd.DataFrame({f'seed_{seeds[i]}': values_a}))

At step 0, with 18
VQE result: 3.35742
Length of this optimization 232, time taken = 1.54 

At step 1, with 28
VQE result: 3.35840
Length of this optimization 229, time taken = 1.488 

At step 2, with 38
VQE result: 3.37012
Length of this optimization 205, time taken = 1.366 

At step 3, with 48
VQE result: 3.35449
Length of this optimization 219, time taken = 1.435 

At step 4, with 58
VQE result: 3.36475
Length of this optimization 195, time taken = 1.257 

At step 5, with 68
VQE result: 3.35938
Length of this optimization 213, time taken = 1.356 

At step 6, with 78
VQE result: 3.37256
Length of this optimization 207, time taken = 1.32 

At step 7, with 88
VQE result: 3.37402
Length of this optimization 195, time taken = 1.357 

At step 8, with 188
VQE result: 3.37842
Length of this optimization 207, time taken = 1.326 



In [22]:
df1 = pd.concat([r_res[i] for i in range(len(r_res))], axis = 1)
df1.to_csv('results_seeds/l2_l05_tl_Ry_c_cobyla_seeds.csv')

In [24]:
r_res=[]
seeds = [18, 28, 38, 48, 58, 68, 78, 88, 188]
for i in range(len(seeds)):
    print(f'At step {i}, with {seeds[i]}')
    counts = []
    values = []
    t0 = time.time()
    result = run_qve_w_specified_optimizer(H4q, COBYLA, ansatz_0d, seeds[i], iterations = 250)
    t1 = time.time()
    print(f'Length of this optimization {len(values)}, time taken = {np.round(t1-t0,3)} \n')
    counts_a = counts
    values_a = values 
    r_res.append(pd.DataFrame({f'seed_{seeds[i]}': values_a}))

At step 0, with 18
VQE result: 3.35742
Length of this optimization 232, time taken = 1.797 

At step 1, with 28
VQE result: 3.35840
Length of this optimization 229, time taken = 1.799 

At step 2, with 38
VQE result: 3.37012
Length of this optimization 205, time taken = 1.604 

At step 3, with 48
VQE result: 3.35449
Length of this optimization 219, time taken = 1.631 

At step 4, with 58
VQE result: 3.36475
Length of this optimization 195, time taken = 1.448 

At step 5, with 68
VQE result: 3.35938
Length of this optimization 213, time taken = 1.529 

At step 6, with 78
VQE result: 3.37256
Length of this optimization 207, time taken = 1.487 

At step 7, with 88
VQE result: 3.37402
Length of this optimization 195, time taken = 1.428 

At step 8, with 188
VQE result: 3.37842
Length of this optimization 207, time taken = 1.479 



In [26]:
df2 = pd.concat([r_res[i] for i in range(len(r_res))], axis = 1)
df2.to_csv('results_seeds/l2_l05_tl_RyY_c_cobyla_seeds.csv')